# TRELLIS.2 — Multi-Image → 3D Asset (Colab A100)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TylerOlszewski/TRELLIS.2/blob/main/notebooks/TRELLIS2_MultiImage_Colab_A100.ipynb)

Generates **one textured 3D asset (GLB + turntable video)** from **multiple photos of the same
object taken from different angles** (no camera poses needed), using
`Trellis2ImageTo3DPipeline.run_multi_image()` from
[TylerOlszewski/TRELLIS.2](https://github.com/TylerOlszewski/TRELLIS.2).

**Requirements**
- Colab **A100 GPU** and the **2025.10 runtime** (Runtime → Change runtime type).
  Pinning the runtime gives the notebook Python 3.12 + PyTorch 2.8 and keeps the compiled
  extension ABI reproducible as Colab's default image changes.
- A Hugging Face account with access to two **gated** models (see the auth cell below).
- Google Drive with a few GB free (used to cache compiled CUDA wheels between sessions).

**Timing**: the first run downloads a prebuilt FlashAttention wheel and compiles the other
CUDA extensions (typically ~15–40 min). Wheels are cached to Drive, so later sessions set
up in a few minutes. Model download and generation time are additional.

Run the cells top to bottom. Setup is idempotent: reconnecting or rerunning a cell reuses
every successfully cached wheel.

In [ ]:
#@title 1. Verify the pinned A100 runtime
!nvidia-smi
import sys, torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU found. In Colab choose Runtime → Change runtime type → A100 GPU.')
name = torch.cuda.get_device_name(0)
runtime_ok = sys.version_info[:2] == (3, 12) and torch.__version__.split('+')[0].startswith('2.8.')
print(f"\nPython {sys.version.split()[0]} | PyTorch {torch.__version__} | CUDA {torch.version.cuda} | GPU: {name}")
if 'A100' not in name:
    raise RuntimeError(f'Expected an A100, but Colab assigned {name}. Change the hardware accelerator to A100.')
if not runtime_ok:
    raise RuntimeError(
        'Select Runtime → Change runtime type → Runtime Version 2025.10, then reconnect. '
        'This notebook intentionally pins Python 3.12 / PyTorch 2.8 for binary compatibility.'
    )
print('✓ compatible Colab A100 runtime')

In [ ]:
#@title 2. Mount Google Drive & set up caches
USE_DRIVE = True  #@param {type:"boolean"}
CACHE_MODELS_ON_DRIVE = False  #@param {type:"boolean"}

# Wheels (slow to build, small) always go to Drive when USE_DRIVE is on.
# Model checkpoints (~15 GB, fast to re-download) only go to Drive if you have the space.
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_ROOT = '/content/drive/MyDrive/TRELLIS2_cache'
else:
    CACHE_ROOT = '/content/TRELLIS2_cache'

WHEELS_DIR = f'{CACHE_ROOT}/wheels'
HF_HOME = f'{CACHE_ROOT}/hf_home' if (USE_DRIVE and CACHE_MODELS_ON_DRIVE) else '/content/hf_home'
os.makedirs(WHEELS_DIR, exist_ok=True)
os.makedirs(HF_HOME, exist_ok=True)
os.environ['HF_HOME'] = HF_HOME
print('wheel cache :', WHEELS_DIR)
print('HF cache    :', HF_HOME)

In [ ]:
#@title 3. Prepare the CUDA build environment and wheel cache
# The pinned 2025.10 runtime has PyTorch/CUDA 12.6 and nvcc 12.5. PyTorch accepts a
# same-major minor-version difference when compiling extensions; a major mismatch is unsafe.
import os, re, shutil, subprocess, sys, pathlib, torch

def _versions():
    nvcc_path = shutil.which('nvcc')
    if nvcc_path is None:
        raise RuntimeError('nvcc is missing from this runtime; select Colab runtime 2025.10.')
    nvcc = subprocess.run([nvcc_path, '--version'], capture_output=True, text=True, check=True).stdout
    m = re.search(r'release (\d+)\.(\d+)', nvcc)
    if m is None or torch.version.cuda is None:
        raise RuntimeError('Could not determine the CUDA toolchain versions.')
    return nvcc_path, (int(m.group(1)), int(m.group(2))), tuple(int(x) for x in torch.version.cuda.split('.')[:2])

nvcc_path, (nv_maj, nv_min), (t_maj, t_min) = _versions()
print(f"system nvcc: {nv_maj}.{nv_min} | torch built for CUDA: {t_maj}.{t_min}")

if nv_maj != t_maj:
    raise RuntimeError(
        f'nvcc {nv_maj}.{nv_min} and torch CUDA {t_maj}.{t_min} have different major versions. '
        'Reconnect using Colab runtime 2025.10 instead of compiling an incompatible extension.'
    )
if nv_min != t_min:
    print('ℹ same CUDA major version; the 12.5/12.6 minor difference is expected on this runtime')

# CUDA_HOME must point at the toolkit that owns the active nvcc.
os.environ['CUDA_HOME'] = str(pathlib.Path(nvcc_path).resolve().parents[1])
os.environ['TORCH_CUDA_ARCH_LIST'] = '8.0'  # compile only for the A100 architecture
os.environ['MAX_JOBS'] = str(min(8, os.cpu_count() or 2))

# Invalidate cached wheels when the torch/CUDA/python environment changed — wheels
# compiled against a different torch ABI crash at import time.
abi = int(torch.compiled_with_cxx11_abi())
env_tag = (f"torch{torch.__version__}-cu{torch.version.cuda}-nvcc{nv_maj}.{nv_min}-"
           f"py{sys.version_info.major}.{sys.version_info.minor}-abi{abi}-sm80")
tag_file = pathlib.Path(WHEELS_DIR) / 'env_tag.txt'
if tag_file.exists() and tag_file.read_text().strip() != env_tag:
    print(f"environment changed ({tag_file.read_text().strip()} → {env_tag}); clearing cached wheels…")
    for whl in pathlib.Path(WHEELS_DIR).glob('*.whl'):
        whl.unlink()
tag_file.write_text(env_tag)
print("wheel-cache tag:", env_tag)

In [ ]:
#@title 4. Clone repo & install Python dependencies
import os, subprocess, sys
REPO_URL = 'https://github.com/TylerOlszewski/TRELLIS.2.git'
if not os.path.isdir('/content/TRELLIS.2'):
    !git clone {REPO_URL} /content/TRELLIS.2
else:
    !git -C /content/TRELLIS.2 pull --ff-only
%cd /content/TRELLIS.2

deps = [
    'imageio', 'imageio-ffmpeg', 'tqdm', 'easydict', 'opencv-python-headless',
    'ninja', 'trimesh', 'kornia', 'timm', 'packaging', 'psutil', 'wheel',
    'plyfile', 'zstandard', 'matplotlib', 'huggingface_hub',
    'transformers>=4.56.0,<5',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *deps], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8'],
               check=True)
print('✓ python deps installed')

In [ ]:
#@title 5. Hugging Face login (gated models)
# Two models used by the pipeline are GATED on Hugging Face — request access first
# (accept the license on each model page before running this cell):
#   • https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m   (image encoder)
#   • https://huggingface.co/briaai/RMBG-2.0                            (background removal)
# Then create a *read* token at https://huggingface.co/settings/tokens and add it to
# Colab Secrets (key icon in the left sidebar) under the name HF_TOKEN.
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print('✓ logged in with Colab secret HF_TOKEN')
except Exception:
    print('No HF_TOKEN Colab secret found — falling back to interactive login:')
    login()

In [ ]:
#@title 6. Build / install CUDA extensions (cached on Drive after first run)
# FlashAttention uses an official prebuilt wheel. The five source-built wheels are pinned
# to immutable revisions and cached as each one succeeds. Rerunning this cell is safe.
import glob, os, shutil, subprocess, sys, torch, pathlib

EXT_ROOT = '/tmp/trellis2_extensions'
os.makedirs(EXT_ROOT, exist_ok=True)

# O-Voxel vendors Eigen as a header-only dependency, but the repository checkout
# intentionally leaves that directory empty. Install the distro headers once and
# expose them at the include path used by o-voxel/setup.py.
eigen_headers = pathlib.Path('/usr/include/eigen3/Eigen')
if not eigen_headers.exists():
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'libeigen3-dev'], check=True)
eigen_link = pathlib.Path('/content/TRELLIS.2/o-voxel/third_party/eigen/Eigen')
if not eigen_link.exists():
    eigen_link.symlink_to(eigen_headers, target_is_directory=True)
print('✓ Eigen headers ready for o-voxel')

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', *args], check=True)

def _clone(url, name, ref):
    dst = os.path.join(EXT_ROOT, name)
    if os.path.exists(dst) and not os.path.isdir(os.path.join(dst, '.git')):
        shutil.rmtree(dst)  # discard only an incomplete clone in the ephemeral /tmp tree
    if not os.path.exists(dst):
        subprocess.run(['git', 'clone', '--recursive', url, dst], check=True)
    subprocess.run(['git', '-C', dst, 'checkout', '--detach', ref], check=True)
    subprocess.run(['git', '-C', dst, 'submodule', 'update', '--init', '--recursive'], check=True)
    return dst

def _wheel(src):
    _pip('wheel', '--no-build-isolation', '--no-deps', '-w', WHEELS_DIR, src)

def _download(url):
    _pip('download', '--no-deps', '-d', WHEELS_DIR, url)

def ensure(import_name, wheel_prefix, build):
    try:
        __import__(import_name)
        print(f'✓ {import_name} already working')
        return
    except Exception:
        pass
    cached = sorted(glob.glob(f'{WHEELS_DIR}/{wheel_prefix}-*.whl'))
    if cached:
        print(f'→ installing {import_name} from cached wheel: {os.path.basename(cached[-1])}')
        _pip('install', '-q', '--force-reinstall', '--no-deps', cached[-1])
        try:
            __import__(import_name)
            print(f'✓ {import_name} (from cache)')
            return
        except Exception as e:
            print(f'  cached wheel unusable ({type(e).__name__}) — replacing it…')
            for path in cached:
                os.unlink(path)
    print(f'→ preparing {import_name} (source builds can take a while)…')
    build()
    built = sorted(glob.glob(f'{WHEELS_DIR}/{wheel_prefix}-*.whl'))
    assert built, f'build produced no wheel matching {wheel_prefix}-*.whl in {WHEELS_DIR}'
    _pip('install', '-q', '--force-reinstall', '--no-deps', built[-1])
    __import__(import_name)
    print(f'✓ {import_name} (installed and cached)')

ABI_LABEL = 'TRUE' if torch.compiled_with_cxx11_abi() else 'FALSE'
FLASH_WHEEL = (
    'https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/'
    f'flash_attn-2.8.3+cu12torch2.8cxx11abi{ABI_LABEL}-cp312-cp312-linux_x86_64.whl'
)
ensure('flash_attn', 'flash_attn',
       lambda: _download(FLASH_WHEEL))
ensure('nvdiffrast', 'nvdiffrast',
       lambda: _wheel(_clone('https://github.com/NVlabs/nvdiffrast.git', 'nvdiffrast',
                                  '253ac4fcea7de5f396371124af597e6cc957bfae')))
ensure('nvdiffrec_render', 'nvdiffrec_render',
       lambda: _wheel(_clone('https://github.com/JeffreyXiang/nvdiffrec.git', 'nvdiffrec',
                                  'b296927cc7fd01c2ac1087c8065c4d7248f72da4')))
ensure('cumesh', 'cumesh',
       lambda: _wheel(_clone('https://github.com/JeffreyXiang/CuMesh.git', 'CuMesh',
                                  '12289e1062f0603f2f0d0771b02e1395d247f26f')))
ensure('flex_gemm', 'flex_gemm',
       lambda: _wheel(_clone('https://github.com/JeffreyXiang/FlexGEMM.git', 'FlexGEMM',
                                  '6dd94a859c26ee8246888502eada3dd8ad85532e')))
ensure('o_voxel', 'o_voxel',
       lambda: _wheel('/content/TRELLIS.2/o-voxel'))

print('\n✓ all CUDA extensions ready')

In [ ]:
#@title 7. Sanity check
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
%cd /content/TRELLIS.2
import o_voxel
from trellis2.pipelines import Trellis2ImageTo3DPipeline  # prints the sparse backends line
from trellis2.utils import render_utils
from trellis2.renderers import EnvMap
print('✓ TRELLIS.2 imports OK')

## Provide input views

Give **2–4 photos of the same object from different angles** (front / side / back works well).
No camera poses are needed. Backgrounds are removed automatically unless you disable
preprocessing. Views that contradict each other geometrically will average into mush —
prefer consistent lighting and the same object state in every shot.

In [ ]:
#@title 8. Upload or select images
IMAGE_SOURCE = 'upload'  #@param ["upload", "drive_folder"]
DRIVE_FOLDER = '/content/drive/MyDrive/TRELLIS2_inputs'  #@param {type:"string"}

import os, shutil
SUPPORTED_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.webp')
INPUT_DIR = '/content/input_views'
shutil.rmtree(INPUT_DIR, ignore_errors=True)
os.makedirs(INPUT_DIR)

if IMAGE_SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        if name.lower().endswith(SUPPORTED_EXTENSIONS):
            with open(os.path.join(INPUT_DIR, os.path.basename(name)), 'wb') as f:
                f.write(data)
        else:
            print(f'Ignoring unsupported file: {name}')
else:
    if not os.path.isdir(DRIVE_FOLDER):
        raise FileNotFoundError(f'Drive input folder does not exist: {DRIVE_FOLDER}')
    for name in sorted(os.listdir(DRIVE_FOLDER)):
        if name.lower().endswith(SUPPORTED_EXTENSIONS):
            shutil.copy(os.path.join(DRIVE_FOLDER, name), INPUT_DIR)

IMAGE_PATHS = sorted(
    os.path.join(INPUT_DIR, f) for f in os.listdir(INPUT_DIR)
    if f.lower().endswith(SUPPORTED_EXTENSIONS)
)
assert len(IMAGE_PATHS) >= 2, 'Multi-image generation needs at least two supported images.'
if len(IMAGE_PATHS) > 12:
    print('⚠ More than 12 views: stochastic mode may not use every view in every sampler.')
print(f'{len(IMAGE_PATHS)} view(s) loaded')

from PIL import Image
for path in IMAGE_PATHS:
    with Image.open(path) as image:
        image.verify()
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(IMAGE_PATHS), figsize=(4 * len(IMAGE_PATHS), 4))
axes = [axes] if len(IMAGE_PATHS) == 1 else list(axes)
for ax, p in zip(axes, IMAGE_PATHS):
    ax.imshow(Image.open(p))
    ax.set_title(os.path.basename(p), fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
#@title 9. Load the TRELLIS.2-4B pipeline (~15 GB download on first run)
from trellis2.pipelines import Trellis2ImageTo3DPipeline
if 'pipeline' not in globals():
    pipeline = Trellis2ImageTo3DPipeline.from_pretrained('microsoft/TRELLIS.2-4B')
    pipeline.cuda()  # low-VRAM mode: submodels move to GPU only while in use
print('✓ pipeline ready')

In [ ]:
#@title 10. Generate the 3D asset
MODE = 'stochastic'  #@param ["stochastic", "multidiffusion"]
RESOLUTION = 'default'  #@param ["default", "512", "1024", "1024_cascade", "1536_cascade"]
SEED = 42  #@param {type:"integer"}
PREPROCESS = True  #@param {type:"boolean"}
OUTPUT_NAME = 'trellis2_multiview'  #@param {type:"string"}
# MODE:       'stochastic' cycles through views across denoising steps (fast);
#             'multidiffusion' averages all views at every step (slower, more stable).
# RESOLUTION: 'default' uses the model's config (1024_cascade). Use '512' if you hit OOM.
# PREPROCESS: automatic background removal + recentering. Disable only for clean-alpha inputs.

import os, torch
assert OUTPUT_NAME and os.path.basename(OUTPUT_NAME) == OUTPUT_NAME, 'OUTPUT_NAME must be a plain filename.'
from PIL import Image, ImageOps

views = []
for path in IMAGE_PATHS:
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        mode = 'RGBA' if ('A' in image.getbands() or 'transparency' in image.info) else 'RGB'
        views.append(image.convert(mode).copy())
mesh = pipeline.run_multi_image(
    views,
    seed=SEED,
    mode=MODE,
    pipeline_type=None if RESOLUTION == 'default' else RESOLUTION,
    preprocess_image=PREPROCESS,
)[0]
mesh.simplify(16777216)  # nvdiffrast limit
torch.cuda.empty_cache()
print(f'✓ mesh generated: {mesh.vertices.shape[0]:,} vertices, {mesh.faces.shape[0]:,} faces')

In [ ]:
#@title 11. Render turntable video
import os, cv2, imageio, torch
from trellis2.utils import render_utils
from trellis2.renderers import EnvMap

OUT_DIR = '/content/outputs'
os.makedirs(OUT_DIR, exist_ok=True)

envmap = EnvMap(torch.tensor(
    cv2.cvtColor(cv2.imread('assets/hdri/forest.exr', cv2.IMREAD_UNCHANGED), cv2.COLOR_BGR2RGB),
    dtype=torch.float32, device='cuda'
))
video = render_utils.make_pbr_vis_frames(render_utils.render_video(mesh, envmap=envmap))
video_path = f'{OUT_DIR}/{OUTPUT_NAME}.mp4'
imageio.mimsave(video_path, video, fps=15)

from IPython.display import Video, display
display(Video(video_path, embed=True, width=512))

In [ ]:
#@title 12. Export GLB (and copy results to Drive)
import os, shutil
import o_voxel

glb = o_voxel.postprocess.to_glb(
    vertices          = mesh.vertices,
    faces             = mesh.faces,
    attr_volume       = mesh.attrs,
    coords            = mesh.coords,
    attr_layout       = mesh.layout,
    voxel_size        = mesh.voxel_size,
    aabb              = [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
    decimation_target = 1000000,
    texture_size      = 4096,
    remesh            = True,
    remesh_band       = 1,
    remesh_project    = 0,
    verbose           = True,
)
glb_path = f'{OUT_DIR}/{OUTPUT_NAME}.glb'
glb.export(glb_path, extension_webp=True)
print('✓ exported', glb_path)

if USE_DRIVE:
    drive_out = '/content/drive/MyDrive/TRELLIS2_outputs'
    os.makedirs(drive_out, exist_ok=True)
    shutil.copy(glb_path, drive_out)
    if 'video_path' in globals() and os.path.exists(video_path):
        shutil.copy(video_path, drive_out)
    print('✓ copied results to', drive_out)

from google.colab import files as colab_files
colab_files.download(glb_path)

## Troubleshooting

| Symptom | Fix |
|---|---|
| Runtime verification fails | Choose **A100** and **Runtime Version 2025.10** under Runtime → Change runtime type, then reconnect. |
| `detected CUDA version … mismatches` during a build | A 12.5/12.6 warning is expected; a major-version error means the wrong Colab runtime is selected. |
| Import error / `undefined symbol` from a cached extension | The env tag in cell 3 should auto-clear stale wheels. If not, delete `TRELLIS2_cache/wheels/` in Drive and re-run cells 3→6. |
| `401/403` when downloading models | Accept the licenses for `facebook/dinov3-vitl16-pretrain-lvd1689m` and `briaai/RMBG-2.0` on Hugging Face, and check your `HF_TOKEN` Colab secret. |
| CUDA OOM during generation | Set `RESOLUTION='512'` (or `'1024'`) in cell 10 and re-run. |
| O-Voxel fails with `Eigen/Dense` or `Eigen/Core` not found | Rerun cell 6; it installs `libeigen3-dev` and links the headers into `o-voxel/third_party/eigen/`. |
| Runtime disconnects while compiling | Reconnect and rerun from the top. Cell 6 reuses each wheel already cached on Drive. |
| FlashAttention wheel is rejected | Confirm cell 1 reports Python 3.12, PyTorch 2.8, and runtime 2025.10; the notebook uses the matching official wheel. |

**Re-running on the same session**: cells 8→12 can be re-run freely with new images or
parameters — the pipeline stays loaded.